# ALQAC 2026 — Private Inference
Production notebook: private input → budgeted cached retrieval → inference → validated submission. It never uploads to the leaderboard.

In [ ]:
from pathlib import Path
import shutil
import os

IS_KAGGLE = Path('/kaggle').exists()
IS_COLAB = 'COLAB_RELEASE_TAG' in os.environ
PROJECT_ROOT = Path.cwd().resolve()
assert (PROJECT_ROOT / 'pyproject.toml').exists(), 'Run notebook from repository root'

In [ ]:
%pip install -q -e .

In [ ]:
if IS_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    raw_token = UserSecretsClient().get_secret('ALQAC_TEAM_TOKEN')
elif IS_COLAB:
    from google.colab import userdata
    raw_token = userdata.get('ALQAC_TEAM_TOKEN')
else:
    from dotenv import load_dotenv
    load_dotenv(PROJECT_ROOT / '.env')
    raw_token = os.getenv('ALQAC_TEAM_TOKEN')
token = (raw_token or '').strip()
assert token, 'Missing ALQAC_TEAM_TOKEN'
os.environ['ALQAC_TEAM_TOKEN'] = token
print({'secret_loaded': True})

In [ ]:
PRIVATE_INPUT = Path('/kaggle/input/alqac2026-private/private_test.json') if IS_KAGGLE else PROJECT_ROOT / 'data/private/private_test.json'
assert PRIVATE_INPUT.exists(), f'Missing private input: {PRIVATE_INPUT}'
RUN_MODE = 'smoke'  # smoke | full
APPROVED_MAX_NETWORK_CALLS = 4  # review and edit after API preflight
CACHE_SEED_PATH = None  # optional private Kaggle Dataset/file seed
assert RUN_MODE in {'smoke', 'full'}
assert isinstance(APPROVED_MAX_NETWORK_CALLS, int) and APPROVED_MAX_NETWORK_CALLS >= 0
LIMIT = 2 if RUN_MODE == 'smoke' else None
OUTPUT_DIR = Path('/kaggle/working/alqac2026/submissions') if IS_KAGGLE else PROJECT_ROOT / 'submissions'
RUN_DIR = OUTPUT_DIR / f'private_candidate_{RUN_MODE}'
CACHE_DB = Path('/kaggle/working/alqac2026/cache/private_case_api.sqlite') if IS_KAGGLE else PROJECT_ROOT / 'cache/private_case_api.sqlite'
CACHE_DB.parent.mkdir(parents=True, exist_ok=True)
if CACHE_SEED_PATH is not None:
    seed = Path(CACHE_SEED_PATH)
    assert seed.is_file(), f'Cache seed does not exist: {seed}'
    if not CACHE_DB.exists():
        shutil.copy2(seed, CACHE_DB)

In [ ]:
from alqac2026.case_retrieval import SQLiteEvidenceCache, build_api_plan
from alqac2026.config import load_config
from alqac2026.data import load_inference_cases
from alqac2026.runner import run_experiment

config = load_config(PROJECT_ROOT / 'configs/candidate.yaml')
cases = load_inference_cases(PRIVATE_INPUT)
if LIMIT is not None:
    cases = cases[:LIMIT]
cache = SQLiteEvidenceCache(CACHE_DB)
try:
    api_plan = build_api_plan(
        cases,
        cache,
        max_queries=int(config['case_retrieval']['max_queries']),
        approved_max_network_calls=APPROVED_MAX_NETWORK_CALLS,
    )
finally:
    cache.close()
print({key: api_plan[key] for key in ('logical_queries', 'cache_hits', 'cache_misses', 'approved_max_network_calls')})
assert api_plan['cache_misses'] <= APPROVED_MAX_NETWORK_CALLS, api_plan

try:
    result = run_experiment(
        config_path=PROJECT_ROOT / 'configs/candidate.yaml',
        input_path=PRIVATE_INPUT,
        resume_run=RUN_DIR,
        limit=LIMIT,
        cache_db=CACHE_DB,
        max_network_calls=APPROVED_MAX_NETWORK_CALLS,
    )
finally:
    if IS_KAGGLE and CACHE_DB.exists():
        export_path = Path('/kaggle/working/export/private_case_api.sqlite')
        export_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(CACHE_DB, export_path)
        print({'preserve_cache_separately': str(export_path)})
assert result['validation']['status'] == 'PASS'
result